# 42_01 - Setup & partisi multi-klien (Paper 4-2, Fase 1)

**Tujuan:** siapkan partisi **K=10 klien** di ruang **SFM 9-fitur** untuk studi
*robust aggregation*. Memuat partisi SFM Paper 4 (`cic/unsw *_train.npz` dari
`federated/notebooks/fed_out/` atau S3), menggabung, lalu **re-partisi non-IID**
(Dirichlet alpha) menjadi K klien. Sebagian klien nanti ditandai **jahat** (di nb
42_04-42_06); di sini hanya partisi + statistik + simpan.

**Konsistensi:** ruang fitur & z-score mengikuti `../aws/fed_node.py` (reuse).
Partisi K-klien disimpan ke `fed_out42/parts_K10.npz` + statistik CSV.

**Output:** `fed_out42/parts_K10.npz`, `fed_out42/parts_stats.csv`,
`fed_out42/global_test.npz` (held-out gabungan utk evaluasi).

> Jalankan di SageMaker (butuh partisi SFM Paper 4). JANGAN mengarang angka.

## 0. Setup + impor fed_node (ruang/zscore konsisten)

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEED=42; K=10; ALPHA=0.5   # alpha moderat; sweep alpha di nb lanjutan bila perlu
print('fed_node LAYERS', fed.LAYERS, '| K', K, '| alpha', ALPHA)

## 1. Muat partisi SFM Paper 4 (lokal atau S3) + PRASYARAT

Cari `cic_train.npz / unsw_train.npz / *_test.npz` dari Paper 4 (`fed_out/`).
Bila tak ada lokal, unduh dari S3 `unsw-far/federated/data/{cic,unsw}/`.
Hard-stop bila tetap tak lengkap (jangan mengarang).

In [ ]:
P4='../notebooks/fed_out'   # lokasi output Paper 4 nb01
def find_or_s3(fname, sub):
    # 1) lokal Paper 4
    for cand in (os.path.join(P4,fname), os.path.join(OUTDIR,fname)):
        if os.path.exists(cand): return cand
    # 2) S3
    try:
        import boto3; s3=boto3.client('s3',region_name=REGION)
        dst=os.path.join(OUTDIR,fname)
        s3.download_file(S3_BUCKET, f'unsw-far/federated/data/{sub}/{fname}', dst)
        print('unduh S3:', fname); return dst
    except Exception as e:
        print('(S3 gagal utk %s: %s)'%(fname,e)); return None
def load_npz(path):
    if path is None or not os.path.exists(path): return None,None
    d=np.load(path,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
paths={'cic_train':find_or_s3('cic_train.npz','cic'), 'cic_test':find_or_s3('cic_test.npz','cic'),
       'unsw_train':find_or_s3('unsw_train.npz','unsw'), 'unsw_test':find_or_s3('unsw_test.npz','unsw')}
Xc_tr,yc_tr=load_npz(paths['cic_train']); Xc_te,yc_te=load_npz(paths['cic_test'])
Xu_tr,yu_tr=load_npz(paths['unsw_train']); Xu_te,yu_te=load_npz(paths['unsw_test'])
missing=[k for k,v in paths.items() if v is None or not os.path.exists(v)]
assert not missing, ('PRASYARAT GAGAL - partisi SFM Paper 4 tak ditemukan: %s. '
                     'Jalankan Paper 4 nb01 dulu / cek S3 unsw-far/federated/data/.'%missing)
print('CIC train',Xc_tr.shape,'UNSW train',Xu_tr.shape)

## 2. Gabung train + held-out global test

Gabungan train (CIC+UNSW) = kolam data yang akan dipartisi ke K klien.
GLOBAL-test = gabungan test CIC+UNSW (held-out; dipakai evaluasi di semua nb 42).

In [ ]:
Xtr=np.vstack([Xc_tr,Xu_tr]); ytr=np.concatenate([yc_tr,yu_tr])
Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
# tandai asal domain (0=CIC,1=UNSW) utk analisis non-IID lintas-dataset
dom=np.concatenate([np.zeros(len(yc_tr),int), np.ones(len(yu_tr),int)])
print('pool train', Xtr.shape, 'pos', round(float(ytr.mean()),3))
print('GLOBAL-test', Xg_te.shape, 'pos', round(float(yg_te.mean()),3))

## 3. Partisi non-IID ke K klien (Dirichlet berbasis label)

Proporsi kelas tiap klien ditarik dari Dirichlet(alpha). alpha kecil = sangat
non-IID. Catat ukuran, pos-rate, dan komposisi domain tiap klien (apakah klien
cenderung CIC atau UNSW) — relevan utk analisis 'penyerang bersembunyi di non-IID'.

In [ ]:
def dirichlet_partition(X, y, dom, K, alpha, seed=SEED):
    r=np.random.RandomState(seed); idx_by_c={c:np.where(y==c)[0] for c in np.unique(y)}
    for c in idx_by_c: r.shuffle(idx_by_c[c])
    ci=[[] for _ in range(K)]
    for c,idc in idx_by_c.items():
        prop=r.dirichlet([alpha]*K); cuts=(np.cumsum(prop)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc,cuts)): ci[k].extend(part.tolist())
    parts=[]
    for k in range(K):
        ii=np.array(ci[k],int); r.shuffle(ii)
        parts.append(ii)
    return parts
parts_idx=dirichlet_partition(Xtr,ytr,dom,K,ALPHA,seed=SEED)
rows=[]
for k,ii in enumerate(parts_idx):
    n=len(ii); pos=float(ytr[ii].mean()) if n else 0.0
    cic_frac=float((dom[ii]==0).mean()) if n else 0.0
    rows.append({'client':k,'n':n,'pos_rate':round(pos,4),'cic_frac':round(cic_frac,4)})
dfst=pd.DataFrame(rows); dfst.to_csv(os.path.join(OUTDIR,'parts_stats.csv'),index=False)
import IPython.display as ipd; ipd.display(dfst)
degenerate=[r['client'] for r in rows if r['n']<10 or r['pos_rate'] in (0.0,1.0)]
print('klien degenerate (hati-hati):', degenerate if degenerate else 'tidak ada')
print('=== SEL 3 (partisi K klien) SELESAI ===')

## 4. Simpan partisi K-klien + global test (untuk nb 42_02-42_07)

Simpan sebagai object array (indeks per klien) + data pool, agar nb lanjutan
tinggal muat. z-score TETAP per-klien lokal (dihitung saat training, bukan di sini).

In [ ]:
np.savez_compressed(os.path.join(OUTDIR,'parts_K10.npz'),
    X=Xtr.astype(float), y=ytr.astype(int), dom=dom.astype(int),
    parts=np.array([np.asarray(p,int) for p in parts_idx], dtype=object),
    K=np.array(K), alpha=np.array(ALPHA), seed=np.array(SEED))
np.savez_compressed(os.path.join(OUTDIR,'global_test.npz'),
    X=Xg_te.astype(float), y=yg_te.astype(int))
print('saved parts_K10.npz & global_test.npz ->', OUTDIR)
# Unggah ke S3 federated42 (boto3; bukan aws s3 cp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in ('parts_K10.npz','global_test.npz','parts_stats.csv'):
        s3.upload_file(os.path.join(OUTDIR,f),S3_BUCKET,f'unsw-far/federated42/data/{f}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/federated42/data/'%S3_BUCKET)
except Exception as e: print('(upload S3 dilewati:',e,')')
print('=== SEL 4 (simpan partisi) SELESAI ===')

## 5. Catatan untuk naskah

- Partisi K=10 non-IID (Dirichlet alpha=0.5) di ruang SFM 9-fitur; pool = CIC+UNSW.
- `cic_frac` per klien menunjukkan derajat dominasi domain — dipakai menilai apakah
  penyerang mudah 'bersembunyi' di antara klien yang distribusinya ekstrem.
- z-score per-klien lokal (privasi) dihitung saat training di nb lanjutan, bukan di sini.
- Fraksi penyerang f=10/20/30% → 1/2/3 klien jahat pada K=10 (ditetapkan di 42_04-42_06).